<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
텐서(Tensor) 기초
</div>

# 생성 4경로 확인

**목표**: Tensor가 어떤 입구로 만들어졌는지와 metadata가 어떻게 정해지는지 확인합니다.

In [3]:
import numpy as np
import torch

In [4]:
data = [[1, 2], [3, 4]]
np_array = np.array(data)

x_data = torch.tensor(data)
x_np = torch.from_numpy(np_array)
x_ones = torch.ones_like(x_data)
x_rand = torch.rand_like(x_data, dtype=torch.float)
x_shape = torch.zeros((2, 3))

for name, t in [
    ("x_data", x_data),
    ("x_np", x_np),
    ("x_ones", x_ones),
    ("x_rand", x_rand),
    ("x_shape", x_shape),
]:
    print("[+]", name, t)
    print("  shape:", t.shape, "dtype:", t.dtype, "device:", t.device)


[+] x_data tensor([[1, 2],
        [3, 4]])
  shape: torch.Size([2, 2]) dtype: torch.int64 device: cpu
[+] x_np tensor([[1, 2],
        [3, 4]])
  shape: torch.Size([2, 2]) dtype: torch.int64 device: cpu
[+] x_ones tensor([[1, 1],
        [1, 1]])
  shape: torch.Size([2, 2]) dtype: torch.int64 device: cpu
[+] x_rand tensor([[0.3552, 0.3973],
        [0.6133, 0.8449]])
  shape: torch.Size([2, 2]) dtype: torch.float32 device: cpu
[+] x_shape tensor([[0., 0., 0.],
        [0., 0., 0.]])
  shape: torch.Size([2, 3]) dtype: torch.float32 device: cpu


# metadata 3축 출력

**목표**: 오류를 추적하기 전에 항상 `shape`, `dtype`, `device`를 먼저 찍어 봅니다.

In [5]:
def inspect(name, t):
    print(f"{name:10s} shape={tuple(t.shape)} dtype={t.dtype} device={t.device}")

In [6]:
a = torch.tensor([[1, 2, 3], [4, 5, 6]])
b = torch.rand(2, 3)
c = torch.ones((2, 3), dtype=torch.float64)

In [7]:
inspect("a", a)
inspect("b", b)
inspect("c", c)

a          shape=(2, 3) dtype=torch.int64 device=cpu
b          shape=(2, 3) dtype=torch.float32 device=cpu
c          shape=(2, 3) dtype=torch.float64 device=cpu


# 사용 가능한 accelerator로 이동하기

**목표**: CPU에서 만든 Tensor를 사용 가능한 장치로 옮기고, 이동 뒤 `device`가 바뀌는지 확인합니다.

In [8]:
def pick_device():
    if hasattr(torch, "accelerator") and torch.accelerator.is_available():
        return torch.accelerator.current_accelerator()
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [9]:
# GPU/가속기가 있다면 `after`가 `cuda:0`, `mps`, `xpu:0` 같은 값으로 나올 수 있습니다.
device = pick_device()
tensor = torch.rand(3, 4)
print("before:", tensor.device)

tensor = tensor.to(device)
print("after: ", tensor.device)

before: cpu
after:  cpu


In [10]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
device

device(type='cpu')

# indexing, slicing, assignment

**목표**: 행, 열, 마지막 열을 골라 보고, 특정 열 값을 바꿉니다.

In [11]:
tensor = torch.rand(4, 4)
print(tensor)

print("first row:", tensor[0])
print("first column:", tensor[:, 0])
print("last column:", tensor[..., -1])

tensor[:, 1] = 0
print(tensor)


tensor([[0.8579, 0.1011, 0.7158, 0.4193],
        [0.1039, 0.9162, 0.8140, 0.3314],
        [0.9835, 0.8068, 0.5404, 0.2685],
        [0.9894, 0.0075, 0.2622, 0.1041]])
first row: tensor([0.8579, 0.1011, 0.7158, 0.4193])
first column: tensor([0.8579, 0.1039, 0.9835, 0.9894])
last column: tensor([0.4193, 0.3314, 0.2685, 0.1041])
tensor([[0.8579, 0.0000, 0.7158, 0.4193],
        [0.1039, 0.0000, 0.8140, 0.3314],
        [0.9835, 0.0000, 0.5404, 0.2685],
        [0.9894, 0.0000, 0.2622, 0.1041]])


# `cat` & `stack`으로 이어 붙이기

**목표**: 같은 shape의 Tensor를 행 방향 또는 열 방향으로 붙였을 때 결과 shape가 어떻게 바뀌는지 확인합니다.

In [12]:
t1 = torch.ones(2, 3)
t2 = torch.zeros(2, 3)

In [13]:
# cat 사용
cat_rows = torch.cat([t1, t2], dim=0)
cat_cols = torch.cat([t1, t2], dim=1)
print(cat_rows.shape)
print(cat_cols.shape)

torch.Size([4, 3])
torch.Size([2, 6])


In [14]:
# vstack 사용
vstack = torch.vstack([t1, t2])
hstack = torch.hstack([t1, t2])
print(vstack.shape)
print(hstack.shape)

torch.Size([4, 3])
torch.Size([2, 6])


In [15]:
stack = torch.stack([t1, t2])
print(stack.shape)

torch.Size([2, 2, 3])


# `@`와 `*` 비교하기

**목표**: 행렬 곱과 element-wise 곱의 결과 shape 차이를 확인합니다.

In [16]:
a = torch.ones(2, 3)
b = torch.full((3, 2), 2.0) # 원하는 크기의 텐서를 생성하고 모든 원소를 같은 값으로 채웁
c = torch.arange(6, dtype=torch.float32).reshape(2, 3) # 일정한 간격의 숫자들을 가진 1차원 텐서를 만드는 함수
a, b, c

(tensor([[1., 1., 1.],
         [1., 1., 1.]]),
 tensor([[2., 2.],
         [2., 2.],
         [2., 2.]]),
 tensor([[0., 1., 2.],
         [3., 4., 5.]]))

In [17]:
# 행렬곱: a @ b
print(a @ b)
print((a @ b).shape)

tensor([[6., 6.],
        [6., 6.]])
torch.Size([2, 2])


In [18]:
# 원소별곱: a * c
print(a * c)
print((a * c).shape)

tensor([[0., 1., 2.],
        [3., 4., 5.]])
torch.Size([2, 3])


In [19]:
out = torch.empty(2, 2)
torch.matmul(a, b, out=out)
out

tensor([[6., 6.],
        [6., 6.]])

# reduction과 `.item()` 확인

**목표**: 여러 값을 하나로 줄인 뒤 Python 숫자로 꺼내는 흐름을 확인합니다.

In [20]:
x = torch.arange(1, 7, dtype=torch.float32).reshape(2, 3)
x

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [21]:
total = x.sum()
row_sum = x.sum(dim=1)

print(total, total.shape)
print(total.item())
print(row_sum, row_sum.shape)

tensor(21.) torch.Size([])
21.0
tensor([ 6., 15.]) torch.Size([2])


# in-place 연산 관찰하기

**목표**: `_`가 붙은 연산이 원본 Tensor를 직접 바꾸는지 확인합니다.

In [22]:
x = torch.ones(2, 2)
y = x.add(5)

print("after add")
print("x =", x)
print("y =", y)

x.add_(5) #값을 바로 고칠 대 사용, 단 학습하는 과정에서 사용 금지
print("after add_")
print("x =", x)

after add
x = tensor([[1., 1.],
        [1., 1.]])
y = tensor([[6., 6.],
        [6., 6.]])
after add_
x = tensor([[6., 6.],
        [6., 6.]])


# Tensor에서 NumPy로 공유 확인

**목표**: CPU Tensor를 NumPy 배열로 바꾸었을 때 같은 메모리를 공유하는지 확인합니다.

In [23]:
t = torch.ones(5)
n = t.numpy()

print("before")
print(t)
print(n)

n[0] = 99 #n만 변경했지만 tensor에서도 변경

print("after")
print(t)
print(n)

before
tensor([1., 1., 1., 1., 1.])
[1. 1. 1. 1. 1.]
after
tensor([99.,  1.,  1.,  1.,  1.])
[99.  1.  1.  1.  1.]


# NumPy에서 Tensor로 공유 확인

**목표**: `torch.from_numpy`도 메모리를 공유할 수 있음을 확인합니다.

In [24]:
arr = np.ones(5)
t = torch.from_numpy(arr)

t[1] = 77
t, arr

(tensor([ 1., 77.,  1.,  1.,  1.], dtype=torch.float64),
 array([ 1., 77.,  1.,  1.,  1.]))

In [25]:
safe_t = torch.from_numpy(arr).clone()
safe_t[2] = 55
print(arr)
print(safe_t)

[ 1. 77.  1.  1.  1.]
tensor([ 1., 77., 55.,  1.,  1.], dtype=torch.float64)
